In [27]:
import cellxgene_census
import pandas as pd
import scanpy as sc
import numpy as np
import json
import anndata as ad
import re


In [28]:
cell_gene_dir = "../../data/censusxgene"
# cell_types = [
#     "malignant cell",
#     "luminal epithelial cell of mammary gland",
#     "basal-myoepithelial cell of mammary gland",
#     "fibroblast of mammary gland",
#     "macrophage",
#     "T cell",
#     "B cell"
# ]

tissues = ['breast', 'lung', 'kidney', 'bladder organ'] # 15M cells
tissues_general = ['breast', 'lung', 'kidney', 'bladder organ'] # 17.9M cells
# simplified & only cell types that are in all tissues - # 12M cells
# only cell types that are in all tissueas - # 5M cells

with open("../../data/protein_coding_genes.txt", "r") as f:
    protein_coding_genes = [line.strip() for line in f]

In [50]:
with cellxgene_census.open_soma(census_version="2025-11-08") as census:
    obs_df = cellxgene_census.get_obs(
        census,
        "homo_sapiens",
        value_filter=f"tissue_general in {tissues_general} and suspension_type == 'cell'" # and cell_type in {cell_types}"
    )

In [51]:
obs_df

,soma_joinid,dataset_id,assay,assay_ontology_term_id,cell_type,cell_type_ontology_term_id,development_stage,development_stage_ontology_term_id,disease,disease_ontology_term_id,...,tissue,tissue_ontology_term_id,tissue_type,tissue_general,tissue_general_ontology_term_id,raw_sum,nnz,raw_mean_nnz,raw_variance_nnz,n_measured_vars
0,31188,703f00e6-b996-48e5-bc34-00c41b9876f4,10x 5' v1,EFO:0011025,Schwann cell,CL:0002573,18th week post-fertilization stage,HsapDv:0000055,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,5780.0,2236,2.584973,42.338638,16035
1,31189,703f00e6-b996-48e5-bc34-00c41b9876f4,10x 5' v1,EFO:0011025,Schwann cell,CL:0002573,15th week post-fertilization stage,HsapDv:0000052,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,2087.0,1357,1.537951,5.754643,16035
2,31190,703f00e6-b996-48e5-bc34-00c41b9876f4,10x 5' v1,EFO:0011025,Schwann cell,CL:0002573,15th week post-fertilization stage,HsapDv:0000052,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,14733.0,4268,3.451968,107.457969,16035
3,31191,703f00e6-b996-48e5-bc34-00c41b9876f4,10x 5' v1,EFO:0011025,immature Schwann cell,CL:0002377,Carnegie stage 19,HsapDv:0000026,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,5174.0,1797,2.879243,53.412470,16035
4,31192,703f00e6-b996-48e5-bc34-00c41b9876f4,10x 5' v1,EFO:0011025,neuron,CL:0000540,Carnegie stage 19,HsapDv:0000026,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,8107.0,3033,2.672931,31.431249,16035
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17890872,156173185,53d208b0-2cfd-4366-9866-c3c6114081bc,10x 3' v3,EFO:0009922,"CD8-positive, alpha-beta T cell",CL:0000625,60-year-old stage,HsapDv:0000154,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,4561.0,2016,2.262401,53.842278,60606
17890873,156173186,53d208b0-2cfd-4366-9866-c3c6114081bc,10x 3' v3,EFO:0009922,capillary endothelial cell,CL:0002144,60-year-old stage,HsapDv:0000154,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,8823.0,3366,2.621212,68.019033,60606
17890874,156173187,53d208b0-2cfd-4366-9866-c3c6114081bc,10x 3' v3,EFO:0009922,endothelial cell of artery,CL:1000413,60-year-old stage,HsapDv:0000154,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,12442.0,3943,3.155465,133.749290,60606
17890875,156173188,53d208b0-2cfd-4366-9866-c3c6114081bc,10x 3' v3,EFO:0009922,macrophage,CL:0000235,60-year-old stage,HsapDv:0000154,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,38498.0,6121,6.289495,994.178270,60606


In [52]:
cell_types = set(obs_df['cell_type'].values)
len(cell_types)


360

In [53]:
def simplify_cell_type(x):
    x_lower = x.lower()

    # --- T CELLS ---
    if "t cell" in x_lower:
        return "T cell"

    # --- B CELLS ---
    if "b cell" in x_lower or "plasma" in x_lower:
        return "B cell"

    # --- NK ---
    if "natural killer" in x_lower or "nk" in x_lower:
        return "NK cell"

    # --- MACROPHAGE ---
    if "macrophage" in x_lower:
        return "macrophage"

    # --- DENDRITIC ---
    if "dendritic" in x_lower:
        return "dendritic cell"

    # --- MONOCYTE / MYELOID ---
    if "monocyte" in x_lower or "myeloid" in x_lower:
        return "myeloid cell"

    # --- ENDOTHELIAL ---
    if "endothelial" in x_lower:
        return "endothelial cell"

    # --- EPITHELIAL ---
    if "epithelial" in x_lower or "epithelium" in x_lower:
        return "epithelial cell"

    # --- FIBROBLAST / STROMAL ---
    if "fibroblast" in x_lower:
        return "fibroblast"
    
    if "stromal" in x_lower or "mesenchymal" in x_lower or "pericyte" in x_lower:
        return "stromal cell"

    # --- MUSCLE ---
    if "muscle" in x_lower:
        return "muscle cell"

    # --- NEURON / GLIA ---
    if "neuron" in x_lower or "glial" in x_lower or "schwann" in x_lower:
        return "neural cell"

    # --- SECRETORY ---
    if "goblet" in x_lower or "secretory" in x_lower or "acinar" in x_lower:
        return "secretory cell"

    # --- STEM / PROGENITOR ---
    if "stem" in x_lower or "progenitor" in x_lower or "precursor" in x_lower:
        return "progenitor cell"

    # --- BLOOD ---
    if "erythro" in x_lower:
        return "erythroid cell"
    
    if "platelet" in x_lower or "megakaryocyte" in x_lower:
        return "megakaryocyte"

    # --- IMMUNE (fallback) ---
    if "lymphocyte" in x_lower or "leukocyte" in x_lower:
        return "immune cell"

    return x  # zostaw jeśli nie pasuje

In [54]:
# 1. upewnij się, że pracujesz na stringach
obs_df["cell_type"] = obs_df["cell_type"].astype(str)
obs_df["tissue_general"] = obs_df["tissue_general"].astype(str)

# 2. policz tissues per cell_type
counts = (
    obs_df
    .groupby("cell_type", observed=True)["tissue_general"]
    .nunique()
    .reset_index(name="n_tissues")
)

# 3. wybierz rzadkie typy
rare_types = set(counts[counts["n_tissues"] < 4]["cell_type"])

# 4. domyślnie kopiujemy
obs_df["cell_type_coarse"] = obs_df["cell_type"]

# 5. maska tylko dla rzadkich
mask = obs_df["cell_type"].isin(rare_types)

# 6. mapping tylko dla rzadkich
obs_df.loc[mask, "cell_type_coarse"] = (
    obs_df.loc[mask, "cell_type"].apply(simplify_cell_type)
)

In [ ]:
# counts = (
#     obs_df
#     .groupby(["tissue_general", "tissue", "cell_type"], observed=True)
#     .size()
#     .reset_index(name="count")
# )

# counts.to_csv(f"{cell_gene_dir}/cell_counts_per_group.csv", index=False)

In [71]:
# obs_df.groupby("cell_type_coarse")["tissue_general"].nunique().sort_values(ascending=False)
counts = (
    obs_df
    .groupby("cell_type_coarse", observed=True)["tissue_general"]
    .nunique()
    .sort_values(ascending=False)
)

# # counts.to_csv("cell_type_across_tissues_nunique_simplified.csv")

In [72]:
counts

cell_type_coarse
T cell                               4
endothelial cell                     4
regulatory T cell                    4
plasma cell                          4
pericyte                             4
                                    ..
tracheobronchial serous cell         1
urothelial cell                      1
vasa recta cell                      1
kidney arterial blood vessel cell    1
kidney cell                          1
Name: tissue_general, Length: 129, dtype: int64

In [73]:
valid_types = counts[counts == 4].index
valid_types

Index(['T cell', 'endothelial cell', 'regulatory T cell', 'plasma cell',
       'pericyte', 'dendritic cell', 'neutrophil', 'endocrine cell',
       'capillary endothelial cell', 'B cell',
       'CD4-positive, alpha-beta T cell', 'CD8-positive, alpha-beta T cell',
       'mast cell', 'epithelial cell', 'monocyte', 'fibroblast',
       'myofibroblast cell', 'natural killer cell',
       'endothelial cell of lymphatic vessel', 'muscle cell', 'macrophage',
       'mature NK T cell', 'stromal cell'],
      dtype='object', name='cell_type_coarse')

In [74]:
len(valid_types)

23

In [75]:
filtered_df = obs_df[obs_df["cell_type_coarse"].isin(valid_types)]
filtered_df.shape


(11978938, 29)

In [76]:
counts = (
    filtered_df
    .groupby(["tissue_general", "cell_type_coarse"], observed=True)
    .size()
    .reset_index(name="count")
)

# counts = counts[counts["count"] > 1000]
# counts = counts[counts["cell_type"] != 'unknown']

# counts.to_csv(f"{cell_gene_dir}/cell_counts_per_group_all_tissues.csv", index=False)

In [77]:
counts

,tissue_general,cell_type_coarse,count
0,bladder organ,B cell,1475
1,bladder organ,"CD4-positive, alpha-beta T cell",6567
2,bladder organ,"CD8-positive, alpha-beta T cell",13296
3,bladder organ,T cell,2448
4,bladder organ,capillary endothelial cell,321
...,...,...,...
87,lung,neutrophil,61045
88,lung,pericyte,24209
89,lung,plasma cell,100734
90,lung,regulatory T cell,70225


In [78]:
min_cells = 7500
filtered_df = filtered_df[filtered_df["cell_type"].isin(valid_types)]
counts = filtered_df["cell_type"].value_counts()
valid_types = counts[counts >= min_cells].index
filtered_df = filtered_df[filtered_df["cell_type"].isin(valid_types)]

In [79]:
min_donor_samples = 50
counts = filtered_df["donor_id"].value_counts()
valid_donors = counts[counts >= min_donor_samples].index
filtered_df = filtered_df[filtered_df["donor_id"].isin(valid_donors)]
filtered_df["donor_id"] = filtered_df["donor_id"].cat.remove_unused_categories()

In [80]:
filtered_df["donor_id"].value_counts()

donor_id
TSP2                                                                             70800
TSP1                                                                             69774
Leader_Merad_2021_706                                                            35819
TSP25                                                                            34836
Case 4                                                                           34231
                                                                                 ...  
KaraayvazBRCA_PT126                                                                 51
homosapiens_None_2023_None_sikkemalisa_001_d10_1101_2022_03_10_483747Donor_02       50
H27913                                                                              50
homosapiens_None_2023_None_sikkemalisa_002_d10_1101_2022_03_10_483747Donor_02       50
homosapiens_None_2023_None_sikkemalisa_002_d10_1101_2022_03_10_483747S3             50
Name: count, Length: 1767, dtype: 

In [81]:
len(valid_types)

21

In [82]:
cells_per_type = 10000
selected_ids = []

for ct in valid_types:
    subset_ct = filtered_df[filtered_df["cell_type"] == ct]
    
    for tissue in subset_ct["tissue_general"].unique():
        subset = subset_ct[subset_ct["tissue_general"] == tissue]
        
        n_available = len(subset)
        n_sample = min(cells_per_type, n_available)
        
        ids = np.random.choice(subset["soma_joinid"].values, n_sample, replace=False)
        selected_ids.extend(ids)

        print(ct, tissue, n_available, n_sample)

macrophage breast 142503 10000
macrophage kidney 44514 10000
macrophage lung 763936 10000
macrophage bladder organ 9621 9621
fibroblast breast 558064 10000
fibroblast kidney 6174 6174
fibroblast lung 62019 10000
fibroblast bladder organ 41378 10000
CD4-positive, alpha-beta T cell kidney 47191 10000
CD4-positive, alpha-beta T cell breast 48860 10000
CD4-positive, alpha-beta T cell lung 407326 10000
CD4-positive, alpha-beta T cell bladder organ 6567 6567
CD8-positive, alpha-beta T cell kidney 59938 10000
CD8-positive, alpha-beta T cell lung 373087 10000
CD8-positive, alpha-beta T cell breast 22974 10000
CD8-positive, alpha-beta T cell bladder organ 13296 10000
natural killer cell kidney 30227 10000
natural killer cell lung 278842 10000
natural killer cell breast 60731 10000
natural killer cell bladder organ 495 495
T cell breast 115565 10000
T cell kidney 35959 10000
T cell lung 208019 10000
T cell bladder organ 2437 2437
capillary endothelial cell kidney 1702 1702
capillary endothelial 

In [83]:
with cellxgene_census.open_soma(census_version="2025-11-08") as census:
    cell_adata = cellxgene_census.get_anndata(
        census,
        "homo_sapiens",
        obs_coords=selected_ids,
        column_names=["assay", "cell_type", "donor_id", "tissue", "tissue_general", "suspension_type", "disease"]
    )

/tmp/ipykernel_221268/1213773830.py:2: FutureWarning: The argument `column_names` is deprecated and will be removed in a future release. Please use `obs_column_names` and `var_column_names` instead.
  cell_adata = cellxgene_census.get_anndata(
/scratch/2370352/conda/envs/myenv/lib/python3.9/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/scratch/2370352/conda/envs/myenv/lib/python3.9/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [84]:
cols_you_want = [
    "soma_joinid",
    "assay",
    "cell_type",
    "donor_id",
    "tissue",
    "tissue_general",
    "suspension_type",
    "disease"
]

cell_adata.obs = cell_adata.obs[cols_you_want]
cell_adata.obs["cell_type"] = cell_adata.obs["cell_type"].cat.remove_unused_categories()

In [87]:
cell_adata.obs['cell_type'].value_counts()[:25]

cell_type
CD8-positive, alpha-beta T cell         40000
macrophage                              39621
myofibroblast cell                      38863
CD4-positive, alpha-beta T cell         36567
fibroblast                              36174
monocyte                                34749
T cell                                  32437
B cell                                  31475
endothelial cell                        30977
natural killer cell                     30495
mature NK T cell                        27621
mast cell                               26488
epithelial cell                         25222
plasma cell                             24533
pericyte                                24526
capillary endothelial cell              22023
endothelial cell of lymphatic vessel    20642
regulatory T cell                       19574
dendritic cell                          17798
neutrophil                              14040
stromal cell                             7910
Name: count, dtype: int6

In [86]:
protein_set = set(protein_coding_genes)

mask = cell_adata.var["feature_name"].apply(lambda x: x in protein_set)
cell_adata = cell_adata[:, mask].copy()

In [88]:
adata_genes = cell_adata.var['feature_name'].tolist()
len(adata_genes)

19930

In [89]:
sc.pp.normalize_total(cell_adata, target_sum=1e4) #, inplace=False)
sc.pp.log1p(cell_adata)

In [ ]:
# cell_adata.write("../../data/censusxgene/blkb_cell_types_common_across_all.h5ad")

In [90]:
print(cell_adata)
print("obs columns:", cell_adata.obs.columns)
print("var shape:", cell_adata.var.shape)
print("X type:", type(cell_adata.X))

AnnData object with n_obs × n_vars = 581735 × 19930
    obs: 'soma_joinid', 'assay', 'cell_type', 'donor_id', 'tissue', 'tissue_general', 'suspension_type', 'disease'
    var: 'soma_joinid', 'feature_id', 'feature_name', 'feature_type', 'feature_length', 'nnz', 'n_measured_obs'
    uns: 'log1p'
obs columns: Index(['soma_joinid', 'assay', 'cell_type', 'donor_id', 'tissue',
       'tissue_general', 'suspension_type', 'disease'],
      dtype='object')
var shape: (19930, 7)
X type: <class 'scipy.sparse._csr.csr_matrix'>


In [91]:
vocab_path = '/scratch/2370352/my-research/papers/scgpt/save/whole_human/vocab.json'

with open(vocab_path, "r") as f:
    vocab = json.load(f)

model_genes = list(vocab.keys())

print(f"Liczba genów w scGPT vocab: {len(model_genes)}")
print(model_genes[:10])  # przykładowe pierwsze 10 genów

Liczba genów w scGPT vocab: 60697
['RP5-973N23.5', 'RP11-182N22.10', 'CTB-53D8.3', 'RP11-348N17.2', 'RP11-205M20.8', 'RP11-326C3.17', 'RP11-439H13.3', 'RP11-413H22.3', 'GET1-SH3BGR', 'CH17-476P10.1']


In [92]:
# adata_genes = lista genów z adata
# model_genes = lista genów z scGPT vocab

adata_genes = cell_adata.var['feature_name'].tolist()

# zamień na sety dla szybkiego porównania
adata_set = set(adata_genes)
model_set = set(model_genes)

# wspólne geny
common_genes = adata_set & model_set

# geny w adata, których nie ma w scGPT
missing_in_model = adata_set - model_set

# geny w scGPT, których nie ma w adata
missing_in_adata = model_set - adata_set

print(f"Liczba genów w adata: {len(adata_genes)}")
print(f"Liczba genów w scGPT vocab: {len(model_genes)}")
print(f"Liczba genów wspólnych: {len(common_genes)}")
print(f"Liczba genów w adata nie w vocab: {len(missing_in_model)}")
print(f"Liczba genów w vocab nie w adata: {len(missing_in_adata)}")

# przykładowe geny
print("Przykłady genów wspólnych:", list(common_genes)[:10])
print("Przykłady genów w adata ale nie w vocab:", list(missing_in_model)[:10])
print("Przykłady genów w vocab ale nie w adata:", list(missing_in_adata)[:10])

Liczba genów w adata: 19930
Liczba genów w scGPT vocab: 60697
Liczba genów wspólnych: 19193
Liczba genów w adata nie w vocab: 686
Liczba genów w vocab nie w adata: 41504
Przykłady genów wspólnych: ['FUBP1', 'FAM120AOS', 'SLC35G1', 'TBC1D8B', 'SLC11A2', 'TRIM28', 'TSPY8', 'MED24', 'PIM2', 'OR5B21']
Przykłady genów w adata ale nie w vocab: ['ENSG00000285779', 'ENSG00000248469', 'ENSG00000273171', 'ENSG00000271793', 'ENSG00000285733', 'ENSG00000268790', 'ENSG00000286022', 'ENSG00000280071', 'ENSG00000285130', 'ENSG00000258945']
Przykłady genów w vocab ale nie w adata: ['RP11-672A2.4', 'NUP50P1', 'HNRNPA1P42', 'RP11-358L22.2', 'RPL23AP22', 'RPS3AP25', 'RP1-47M23.3', 'Y_RNA_ENSG00000252615', 'CH507-39O4.2', 'RNU6-516P']


In [93]:
gene_info = pd.read_csv("/scratch/2370352/my-research/data/gene_info_table.csv")  # lub pełna ścieżka

# Stwórz słownik: ensembl_id -> gene_name
ensg_to_symbol = dict(zip(gene_info['ensembl_id'], gene_info['gene_name']))

print(list(ensg_to_symbol.items())[:10])

[('ENSG00000000003', 'TSPAN6'), ('ENSG00000000005', 'TNMD'), ('ENSG00000000419', 'DPM1'), ('ENSG00000000457', 'SCYL3'), ('ENSG00000000460', 'C1orf112'), ('ENSG00000000938', 'FGR'), ('ENSG00000000971', 'CFH'), ('ENSG00000001036', 'FUCA2'), ('ENSG00000001084', 'GCLC'), ('ENSG00000001167', 'NFYA')]


In [94]:
# jeśli w adata masz geny zapisane jako ENSG, mapujemy je
mapped_genes = []

for g in cell_adata.var['feature_name']:
    if g in ensg_to_symbol:
        mapped_genes.append(ensg_to_symbol[g])
    else:
        mapped_genes.append(g)  # zachowaj tak jak jest, np. już symboliczny gen

# podmieniamy w adata.var
cell_adata.var['feature_name_mapped'] = mapped_genes

In [95]:
# adata_genes = lista genów z adata
# model_genes = lista genów z scGPT vocab

adata_genes = cell_adata.var['feature_name_mapped'].tolist()

# zamień na sety dla szybkiego porównania
adata_set = set(adata_genes)
model_set = set(model_genes)

# wspólne geny
common_genes = adata_set & model_set

# geny w adata, których nie ma w scGPT
missing_in_model = adata_set - model_set

# geny w scGPT, których nie ma w adata
missing_in_adata = model_set - adata_set

print(f"Liczba genów w adata: {len(adata_genes)}")
print(f"Liczba genów w scGPT vocab: {len(model_genes)}")
print(f"Liczba genów wspólnych: {len(common_genes)}")
print(f"Liczba genów w adata nie w vocab: {len(missing_in_model)}")
print(f"Liczba genów w vocab nie w adata: {len(missing_in_adata)}")

# przykładowe geny
print("Przykłady genów wspólnych:", list(common_genes)[:10])
print("Przykłady genów w adata ale nie w vocab:", list(missing_in_model)[:10])
print("Przykłady genów w vocab ale nie w adata:", list(missing_in_adata)[:10])

Liczba genów w adata: 19930
Liczba genów w scGPT vocab: 60697
Liczba genów wspólnych: 19512
Liczba genów w adata nie w vocab: 309
Liczba genów w vocab nie w adata: 41185
Przykłady genów wspólnych: ['FUBP1', 'FAM120AOS', 'SLC35G1', 'TBC1D8B', 'SLC11A2', 'TRIM28', 'TSPY8', 'MED24', 'PIM2', 'OR5B21']
Przykłady genów w adata ale nie w vocab: ['AC007204.1', 'AC058822.1', 'AL035699.1', 'AL035078.4', 'AC010197.2', 'AC019117.3', 'POLR2J3', 'AC063943.2', 'AC002553.1', 'AC025283.2']
Przykłady genów w vocab ale nie w adata: ['RP11-672A2.4', 'NUP50P1', 'HNRNPA1P42', 'RP11-358L22.2', 'RPL23AP22', 'RPS3AP25', 'RP1-47M23.3', 'Y_RNA_ENSG00000252615', 'CH507-39O4.2', 'RNU6-516P']


In [96]:
np.random.seed(42)

train_indices = []
test_indices = []

group_cols = ["tissue_general", "cell_type"]

for _, group_df in cell_adata.obs.groupby(group_cols):
    
    # unikalni donorzy w tej grupie
    donors = group_df["donor_id"].unique()
    donors = np.array(donors)
    np.random.shuffle(donors)
    
    n_donors = len(donors)
    split = int(0.9 * n_donors)
    
    train_donors = set(donors[:split])
    test_donors  = set(donors[split:])
    
    # indeksy komórek
    train_idx = group_df[group_df["donor_id"].isin(train_donors)].index
    test_idx  = group_df[group_df["donor_id"].isin(test_donors)].index
    
    train_indices.extend(train_idx)
    test_indices.extend(test_idx)

# usuń duplikaty (bo donor może być w wielu grupach!)
train_indices = np.unique(train_indices)
test_indices  = np.unique(test_indices)

# usuń ewentualne przecięcia (bezpieczeństwo)
test_indices = np.setdiff1d(test_indices, train_indices)

# subset
train_adata = cell_adata[train_indices].copy()
test_adata  = cell_adata[test_indices].copy()

/tmp/ipykernel_221268/2181996319.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for _, group_df in cell_adata.obs.groupby(group_cols):


In [97]:
train_adata.obs.groupby(["tissue_general", "cell_type"]).size().describe()
test_adata.obs.groupby(["tissue_general", "cell_type"]).size().describe()

/tmp/ipykernel_221268/3973098360.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train_adata.obs.groupby(["tissue_general", "cell_type"]).size().describe()
/tmp/ipykernel_221268/3973098360.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  test_adata.obs.groupby(["tissue_general", "cell_type"]).size().describe()


count      84.000000
mean      817.261905
std       677.298979
min         0.000000
25%       208.500000
50%       787.500000
75%      1197.000000
max      3952.000000
dtype: float64

In [98]:
train_adata.write("/scratch/2370352/my-research/adapter_premium/data_new/blkb_common_fpc_train.h5ad")
test_adata.write("/scratch/2370352/my-research/adapter_premium/data_new/blkb_common_fpc_test.h5ad")


In [85]:
adata = ad.read_h5ad("/scratch/2370352/my-research/adapter_premium/data_new/blkb_common_test.h5ad")


In [86]:
adata.obs['cell_type'].value_counts()

cell_type
mature NK T cell                        8080
myofibroblast cell                      6167
CD8-positive, alpha-beta T cell         4368
dendritic cell                          3904
macrophage                              3878
plasma cell                             3490
monocyte                                3091
mast cell                               3063
epithelial cell                         2950
T cell                                  2948
endothelial cell of lymphatic vessel    2551
natural killer cell                     2500
endothelial cell                        2490
regulatory T cell                       2415
B cell                                  2348
pericyte                                2315
fibroblast                              2108
CD4-positive, alpha-beta T cell         2094
capillary endothelial cell              1557
neutrophil                              1057
stromal cell                             946
Name: count, dtype: int64

In [2]:
adata_test = ad.read_h5ad("/scratch/2370352/my-research/adapter_premium/data_new/blkb_common_train.h5ad")
adata_test.obs['cell_type'].value_counts()


cell_type
macrophage                              35743
CD8-positive, alpha-beta T cell         35632
CD4-positive, alpha-beta T cell         34473
fibroblast                              34066
myofibroblast cell                      32696
monocyte                                31658
T cell                                  29489
B cell                                  29127
endothelial cell                        28487
natural killer cell                     27995
mast cell                               23425
epithelial cell                         22272
pericyte                                22211
plasma cell                             21043
capillary endothelial cell              20466
mature NK T cell                        19541
endothelial cell of lymphatic vessel    18091
regulatory T cell                       17159
dendritic cell                          13894
neutrophil                              12983
stromal cell                             6964
Name: count, dtype: int6

In [4]:
len(adata_test)

517415

In [89]:
cell_adata.obs['donor_id'].value_counts()
cell_adata.obs['donor_id'].value_counts().to_csv("donor_counts_all.csv")